# Library Concurrency Experiment

This notebook runs the **baseline**, **pessimistic**, and **optimistic** tests from one place.

For each strategy, it rebuilds `library_sim` from the clean `library` database so every test starts from the same data. The baseline uses the same total number of operations as the threaded tests:

`TOTAL_OPERATIONS = NUM_THREADS × OPERATIONS_PER_THREAD`


## 1. Imports

In [1]:
import mysql.connector
import threading
import time
from getpass import getpass


## 2. Connection and workload settings

Change `NUM_THREADS` and `OPERATIONS_PER_THREAD` throughout the project. The baseline automatically uses the matching total number of operations.

In [2]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": getpass("MySQL password: ")
}

NUM_THREADS = 50
OPERATIONS_PER_THREAD = 500
TOTAL_OPERATIONS = NUM_THREADS * OPERATIONS_PER_THREAD

print(f"Threads: {NUM_THREADS}")
print(f"Operations per thread: {OPERATIONS_PER_THREAD}")
print(f"Total operations per strategy: {TOTAL_OPERATIONS:,}")


Threads: 50
Operations per thread: 500
Total operations per strategy: 25,000


## 3. Database connection helper

In [3]:
def get_connection(database=None):
    config = DB_CONFIG.copy()
    if database:
        config["database"] = database
    return mysql.connector.connect(**config)


## 4. Rebuild `library_sim`

This replaces the separate database-dupe SQL file. It drops `library_sim`, recreates the tables from `library`, copies the data, and clears the experiment logs so every strategy starts clean.

In [4]:
def rebuild_library_sim():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SET SESSION restrict_fk_on_non_standard_key = OFF")
    cursor.execute("DROP SCHEMA IF EXISTS `library_sim`")
    cursor.execute("CREATE SCHEMA `library_sim`")

    tables = [
        "Book", "Book_copy", "Member", "Loan",
        "Transaction_log", "Conflict_log", "Deadlock_log"
    ]

    for table in tables:
        cursor.execute(
            f"CREATE TABLE `library_sim`.`{table}` LIKE `library`.`{table}`"
        )
        cursor.execute(
            f"INSERT INTO `library_sim`.`{table}` SELECT * FROM `library`.`{table}`"
        )

    # Experiment logs should start empty for each strategy
    cursor.execute("DELETE FROM `library_sim`.`Transaction_log`")
    cursor.execute("DELETE FROM `library_sim`.`Conflict_log`")
    cursor.execute("DELETE FROM `library_sim`.`Deadlock_log`")

    conn.commit()
    cursor.close()
    conn.close()


## 5. Create the three checkout procedures

Because rebuilding `library_sim` deletes its stored procedures, this cell recreates the baseline, pessimistic, and optimistic procedures automatically.

In [5]:
BASELINE_SQL = """
CREATE PROCEDURE checkout_baseline(IN p_copy_id INT)
BEGIN
    DECLARE v_version INT;

    START TRANSACTION;

    SELECT version INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id;

    INSERT INTO transaction_log
    (operation_type, timestamp, isolation_level, strategy,
     success_flag, conflict_flag, deadlock_flag)
    VALUES
    ('checkout', NOW(), 'READ COMMITTED', 'baseline',
     'TRUE', 'FALSE', 'FALSE');

    COMMIT;
END
"""

PESSIMISTIC_SQL = """
CREATE PROCEDURE checkout_pessimistic(IN p_copy_id INT)
BEGIN
    DECLARE v_version INT;

    START TRANSACTION;

    SELECT version INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id
    FOR UPDATE;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id;

    INSERT INTO transaction_log
    (operation_type, timestamp, isolation_level, strategy,
     success_flag, conflict_flag, deadlock_flag)
    VALUES
    ('checkout', NOW(), 'READ COMMITTED', 'pessimistic',
     'TRUE', 'FALSE', 'FALSE');

    COMMIT;
END
"""

OPTIMISTIC_SQL = """
CREATE PROCEDURE checkout_optimistic(
    IN p_copy_id INT,
    IN p_member_id INT,
    IN p_isolation VARCHAR(45)
)
BEGIN
    DECLARE v_version INT;

    SELECT version INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id
      AND version = v_version;

    IF ROW_COUNT() = 0 THEN
        INSERT INTO conflict_log
        (old_version, attempted_version, timestamp)
        VALUES (v_version, v_version + 1, NOW());

        INSERT INTO transaction_log
        (operation_type, timestamp, isolation_level, strategy,
         success_flag, conflict_flag, deadlock_flag)
        VALUES
        ('checkout', NOW(), p_isolation, 'optimistic',
         'FALSE', 'TRUE', 'FALSE');
    ELSE
        INSERT INTO transaction_log
        (operation_type, timestamp, isolation_level, strategy,
         success_flag, conflict_flag, deadlock_flag)
        VALUES
        ('checkout', NOW(), p_isolation, 'optimistic',
         'TRUE', 'FALSE', 'FALSE');
    END IF;

    COMMIT;
END
"""


def create_checkout_procedures():
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    for name, sql in [
        ("checkout_baseline", BASELINE_SQL),
        ("checkout_pessimistic", PESSIMISTIC_SQL),
        ("checkout_optimistic", OPTIMISTIC_SQL)
    ]:
        cursor.execute(f"DROP PROCEDURE IF EXISTS `{name}`")
        cursor.execute(sql)

    cursor.close()
    conn.close()


## 6. Result helpers

The database log is used to determine **actual** successes and conflicts. This avoids counting an optimistic conflict as a success just because the procedure returned without a Python exception.

In [6]:
def get_db_metrics(strategy):
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s AND success_flag = 'TRUE'
    """, (strategy,))
    successes = cursor.fetchone()[0]

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s AND conflict_flag = 'TRUE'
    """, (strategy,))
    conflicts = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM deadlock_log")
    deadlocks_logged = cursor.fetchone()[0]

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s
    """, (strategy,))
    logged_transactions = cursor.fetchone()[0]

    cursor.close()
    conn.close()

    return successes, conflicts, deadlocks_logged, logged_transactions


## 7. Baseline test

The baseline runs sequentially with **no concurrency-control strategy**. It performs the same total number of checkout operations as each threaded test.

In [7]:
def run_baseline():
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    start_time = time.perf_counter()

    for _ in range(TOTAL_OPERATIONS):
        cursor.callproc("checkout_baseline", [1])

    runtime = time.perf_counter() - start_time

    cursor.close()
    conn.close()

    return runtime


## 8. Pessimistic test

In [8]:
pess_python_metrics = {
    "errors": 0,
    "deadlocks": 0,
    "timeouts": 0
}
metric_lock = threading.Lock()


def pessimistic_worker(thread_id):
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    for _ in range(OPERATIONS_PER_THREAD):
        try:
            cursor.callproc("checkout_pessimistic", [1])
        except Exception as e:
            error_text = str(e)
            with metric_lock:
                pess_python_metrics["errors"] += 1
                if "1205" in error_text:
                    pess_python_metrics["timeouts"] += 1
                if "1213" in error_text:
                    pess_python_metrics["deadlocks"] += 1
            conn.rollback()

    cursor.close()
    conn.close()


def run_pessimistic():
    threads = []
    start_time = time.perf_counter()

    for i in range(NUM_THREADS):
        t = threading.Thread(target=pessimistic_worker, args=(i,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    return time.perf_counter() - start_time


## 9. Optimistic test

In [9]:
opt_python_metrics = {
    "errors": 0,
    "deadlocks": 0,
    "timeouts": 0
}


def optimistic_worker(thread_id):
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    for _ in range(OPERATIONS_PER_THREAD):
        try:
            cursor.callproc(
                "checkout_optimistic",
                [1, thread_id, "READ COMMITTED"]
            )
            conn.commit()
        except Exception as e:
            error_text = str(e)
            with metric_lock:
                opt_python_metrics["errors"] += 1
                if "1205" in error_text:
                    opt_python_metrics["timeouts"] += 1
                if "1213" in error_text:
                    opt_python_metrics["deadlocks"] += 1
            conn.rollback()

    cursor.close()
    conn.close()


def run_optimistic():
    threads = []
    start_time = time.perf_counter()

    for i in range(NUM_THREADS):
        t = threading.Thread(target=optimistic_worker, args=(i,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    return time.perf_counter() - start_time


## 10. Run all three strategies

Each strategy receives a fresh clone of `library`. The checkout procedures are recreated after every clone.

In [10]:
# -------------------------
# BASELINE
# -------------------------
print("Rebuilding library_sim for baseline...")
rebuild_library_sim()
create_checkout_procedures()
baseline_time = run_baseline()
baseline_success, baseline_conflicts, baseline_deadlocks, baseline_logged = get_db_metrics("baseline")

# -------------------------
# PESSIMISTIC
# -------------------------
print("Rebuilding library_sim for pessimistic test...")
rebuild_library_sim()
create_checkout_procedures()

for key in pess_python_metrics:
    pess_python_metrics[key] = 0

pess_time = run_pessimistic()
pess_success, pess_conflicts, pess_deadlocks_logged, pess_logged = get_db_metrics("pessimistic")

# -------------------------
# OPTIMISTIC
# -------------------------
print("Rebuilding library_sim for optimistic test...")
rebuild_library_sim()
create_checkout_procedures()

for key in opt_python_metrics:
    opt_python_metrics[key] = 0

opt_time = run_optimistic()
opt_success, opt_conflicts, opt_deadlocks_logged, opt_logged = get_db_metrics("optimistic")

# -------------------------
# COMPARISON
# -------------------------
baseline_tps = TOTAL_OPERATIONS / baseline_time
pess_tps = TOTAL_OPERATIONS / pess_time
opt_attempt_tps = TOTAL_OPERATIONS / opt_time
opt_success_tps = opt_success / opt_time

print("\n" + "=" * 100)
print("CONCURRENCY TEST RESULTS")
print("=" * 100)
print(f"{'Metric':<30}{'Baseline':<22}{'Pessimistic':<22}{'Optimistic':<22}")
print("-" * 100)
print(f"{'Total Attempts':<30}{TOTAL_OPERATIONS:<22}{TOTAL_OPERATIONS:<22}{TOTAL_OPERATIONS:<22}")
print(f"{'Runtime (seconds)':<30}{baseline_time:<22.4f}{pess_time:<22.4f}{opt_time:<22.4f}")
print(f"{'Actual Successes':<30}{baseline_success:<22}{pess_success:<22}{opt_success:<22}")
print(f"{'Conflicts':<30}{baseline_conflicts:<22}{pess_conflicts:<22}{opt_conflicts:<22}")
print(f"{'Python Exceptions':<30}{0:<22}{pess_python_metrics['errors']:<22}{opt_python_metrics['errors']:<22}")
print(f"{'Deadlocks (1213)':<30}{0:<22}{pess_python_metrics['deadlocks']:<22}{opt_python_metrics['deadlocks']:<22}")
print(f"{'Timeouts (1205)':<30}{0:<22}{pess_python_metrics['timeouts']:<22}{opt_python_metrics['timeouts']:<22}")
print(f"{'Attempt Throughput / sec':<30}{baseline_tps:<22.2f}{pess_tps:<22.2f}{opt_attempt_tps:<22.2f}")
print(f"{'Successful Throughput / sec':<30}{baseline_tps:<22.2f}{pess_success / pess_time:<22.2f}{opt_success_tps:<22.2f}")
print(f"{'Transaction Log Rows':<30}{baseline_logged:<22}{pess_logged:<22}{opt_logged:<22}")
print("=" * 100)


Rebuilding library_sim for baseline...
Rebuilding library_sim for pessimistic test...
Rebuilding library_sim for optimistic test...

CONCURRENCY TEST RESULTS
Metric                        Baseline              Pessimistic           Optimistic            
----------------------------------------------------------------------------------------------------
Total Attempts                25000                 25000                 25000                 
Runtime (seconds)             4.5866                2.5845                2.4075                
Actual Successes              25000                 25000                 517                   
Conflicts                     0                     0                     24483                 
Python Exceptions             0                     0                     0                     
Deadlocks (1213)              0                     0                     0                     
Timeouts (1205)               0                     0         

### How to use this notebook

For a new workload, only change:

```python
NUM_THREADS = ...
OPERATIONS_PER_THREAD = ...
```

The baseline automatically runs the matching total number of operations. Each strategy starts from a fresh `library_sim` clone.